<div style="margin-bottom: 120px;">
    <div style="float:left;">
        <br/>
        <img src="img/udc.png" width="300"/>
    </div>
</div>

<h1 style="color: #d60e8c; text-align:center;">Procesado de datos</h1>


<h1>Contenidos</h1>

<div class="alert alert-block alert-info" 
     style="margin-top: 20px; padding-top:0px; padding-bottom:0px;border: 1px solid #d60e8c; border-radius: 20px; background:transparent;">
    <ol>
        <li><a href="#intro">Introducción</a></li>
        <li><a href="#about_dataset">Descripción del conjunto de datos</a></li>
        <li><a href="#analysis">Lectura y análisis de los datos </a></li>
        <li><a href="#division">División del conjunto de datos en datos en entrada y salida</a></li>
        <li><a href="#split">División de los datos para entrenamiento y prueba</a></li>
        <li><a href="#oversampling">Balanceo de datos</a></li>
        <li><a href="#scale">Normalización</a></li>        
        <li><a href="#pca">PCA</a></li>        
        <li><a href="#classification">Ejercicio</a></li>
    </ol>
</div>
<br>

<a name="intro"></a>
<h1 style="color: #d60e8c;">Introducción</h1>
<hr style="border: 0.5px solid #d60e8c;">

En esta práctica aplicaremos PCA para reducir la dimensionalidad de un conjunto de datos y, a continuación, aplicaremos los algoritmos para clasificación supervisada **SVM** y **k-NN**.

## Instalación de librería adicional:

En este cuaderno utilizaremos la librería **inbalanced-learn**, para instalarla, ejecuta la celda siguiente:

In [ ]:
pip install -U imbalanced-learn

## Importamos las librerías

Comenzamos importando las librerías que utilizaremos:

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn import preprocessing
from sklearn.decomposition import PCA

<a name="about_dataset"></a>
<h1 style="color: #d60e8c;">Descripción del conjunto de datos</h1>
<hr style="border: 0.5px solid #d60e8c;">

Utilizaremos un conjunto de datos sobre diagnóstico de enfermedades del corazón. Todos los atributos tienen valores numéricos. Los datos fueron recolectados 
de la "Cleveland Clinic Foundation". 
<br><br>

<table style="border:1px solid black;">
    <tr>   
        <td style="background: #ddeeff; text-align:center; border:1px solid black;"><b>Número de instancias:</b></td>
        <td style="background: #ddeeff; text-align:center; border:1px solid black;"><b>Valores perdidos:</b></td>
        <td style="background: #ddeeff; text-align:center; border:1px solid black;"><b>Atributos para estudio</b></td>
    </tr>
    <tr>
        <td style="background: white; text-align:center; border:1px solid black;">255</td>
        <td style="background: white; text-align:center; border:1px solid black;">Sí (indicados con "?")</td>    
        <td style="background: white; text-align:center; border:1px solid black;">14</td>                
    </tr>
</table>
<br><br>
El conjunto de datos original contenía 75 atributos pero una selección previa realizada por expertos los ha reducido a 14, que son los que se usan para los experimentos:

<ul>
    <li><b>age</b>: la edad de cada individuo.</li>
    <li><b>sex</b>: el género, usando el formato:
        <ul>                        
            <li>0 = female</li>
            <li>1 = male</li>
        </ul>
    </li>
    <li><b>cp</b>: tipo de dolor en le pecho que experimenta el individuo:
        <ul>
            <li>1 = angina típica</li>
            <li>2 = angina atípica</li>
            <li>3 = dolor no angina</li>
            <li>4 = asintomático</li>
        </ul>
    </li>
    <li><b>trestbps</b>: presión sanguínea en reposo en mmHg</li>
    <li><b>chol</b>: colestoral en suero en mg/dl</li>
    <li><b>fbs</b>: compara el valor de azúcar en la sangre en ayunas de un individuo con 120mg/dl:
        <ul>
            <li> Si es > 120mg/dl -> fbs = 1 </li>
            <li> sino -> fbs = 0
        </ul>
    </li>
    <li><b>restecg</b>: muestra los resultados electrocardiográficos en reposo:
        <ul>
            <li> 0 = normal</li>
            <li> 1 = anormalidad en la onda ST-T</li>
            <li> 2 = hipertrofia del ventrículo izquierdo</li>
        </ul>
    </li>
    <li><b>thalach</b> (frecuencia cardíaca máxima alcanzada): </li>
    <li><b>exang</b>: angina inducida por el ejercicio (1 = sí; 0 = no)</li>
    <li><b>oldpeak</b>: la depresión ST inducida por el ejercicio en relación con el descanso</li>
    <li><b>slope</b>: la pendiente del segmento ST del ejercicio máximo:
        <ul>
            <li> 1 = ascenso </li>
            <li> 2 = plano </li>
            <li> 3 = descenso </li>
        </ul>
    </li>    
    <li><b>ca</b>: Número de vasos principales (0-3) coloreados por flourosopía</li>
    <li><b>thal</b>: indica si hay talasemia :
        <ul>
            <li>3 = normal</li>
            <li>6 = fija </li>
            <li>7 = reversible</li>
        </ul>
    </li>    
    <li><b>class</b>: contiene el diagnóstico, indica si el individuo sufre de enfermedad cardíaca o no:
        <ul>
            <li>0: no existe ninguna enfermedad cardíaca</li>
            <li>1: existe enfermedad cardíaca </li>
        </ul>
    </li>
</ul>

<a name="analysis"></a>
<h1 style="color: #d60e8c;">Lectura y análisis de los datos</h1>
<hr style="border: 0.5px solid #d60e8c;">

El archivo con el conjunto de datos no contiene una cabecera con los identificadores de las columnas, con lo que, crearemos una lista con estos identificadores para añadirla al <code>DataFrame</code> cuando leemos el archivo: <code>names=dataLabels</code> (donde dataLabels es la lista creada con los identificadores de las columnas).

Por otro lado, como hay valores perdidos en el dataset, con lo que, cuando leemos el archivo, indicamos que existen valores perdidos con <code>na_values=["?"]</code> (de esta forma, estamos indicando que los valores perdidos están representados por el carácter "?"). Así, en la lectura del archivo, esos valores las celdas que contengan el valor "?" serán convertidas a valores nulos, posteriormente los filtraremos.

In [4]:
df_orig=pd.read_csv('mushroms1.csv', na_values=["?"])
df_orig.head()

,poisonous,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,EDIBLE,CONVEX,SMOOTH,WHITE,BRUISES,ALMOND,FREE,CROWDED,NARROW,WHITE,...,SMOOTH,WHITE,WHITE,PARTIAL,WHITE,ONE,PENDANT,PURPLE,SEVERAL,WOODS
1,EDIBLE,CONVEX,SMOOTH,WHITE,BRUISES,ALMOND,FREE,CROWDED,NARROW,WHITE,...,SMOOTH,WHITE,WHITE,PARTIAL,WHITE,ONE,PENDANT,BROWN,SEVERAL,WOODS
2,EDIBLE,CONVEX,SMOOTH,WHITE,BRUISES,ALMOND,FREE,CROWDED,NARROW,PINK,...,SMOOTH,WHITE,WHITE,PARTIAL,WHITE,ONE,PENDANT,PURPLE,SEVERAL,WOODS
3,EDIBLE,CONVEX,SMOOTH,WHITE,BRUISES,ALMOND,FREE,CROWDED,NARROW,PINK,...,SMOOTH,WHITE,WHITE,PARTIAL,WHITE,ONE,PENDANT,BROWN,SEVERAL,WOODS
4,EDIBLE,CONVEX,SMOOTH,WHITE,BRUISES,ALMOND,FREE,CROWDED,NARROW,BROWN,...,SMOOTH,WHITE,WHITE,PARTIAL,WHITE,ONE,PENDANT,PURPLE,SEVERAL,WOODS


## Eliminación de casos con información incompleta

1. Consultamos las dimensiones del conjunto de datos:

In [5]:
df_orig.shape

(8417, 23)

2. Obtenemos los valores nulos que tenememos en cada columna: sabemos que existen porque en la lectura del archivo, hemos indicado que el conjunto de datos contiene valores perdidos para que los convirtiese en nulos en la lectura. Vemos que tenemos dos valores nulos en la columna **ca** y cuatro en la columna **thal**.

In [6]:
df_orig.isnull().sum()

poisonous                      0
cap-shape                      1
cap-surface                    1
cap-color                      1
bruises                        1
odor                           1
gill-attachment                1
gill-spacing                   1
gill-size                      1
gill-color                     1
stalk-shape                    1
stalk-root                  2481
stalk-surface-above-ring       1
stalk-surface-below-ring       1
stalk-color-above-ring         1
stalk-color-below-ring         1
veil-type                      1
veil-color                     1
ring-number                    1
ring-type                      1
spore-print-color              1
population                     1
habitat                        1
dtype: int64

3. Eliminamos las filas que contienen algún valor nulo en alguna de sus columnas, ya que tienen información incompleta.

In [7]:
df=df_orig.dropna(axis=0)

4. Consultamos las dimensiones del dataset tras eliminar las filas con valores nulos.

In [8]:
df.shape

(5936, 23)

5. Consultamos de nuevo los valores nulos para verificar que ya no existen. Ahora el resultado debería ser 0 para todas las columnas.

In [9]:
df.isnull().sum()

poisonous                   0
cap-shape                   0
cap-surface                 0
cap-color                   0
bruises                     0
odor                        0
gill-attachment             0
gill-spacing                0
gill-size                   0
gill-color                  0
stalk-shape                 0
stalk-root                  0
stalk-surface-above-ring    0
stalk-surface-below-ring    0
stalk-color-above-ring      0
stalk-color-below-ring      0
veil-type                   0
veil-color                  0
ring-number                 0
ring-type                   0
spore-print-color           0
population                  0
habitat                     0
dtype: int64

In [10]:
df['poisonous'].value_counts()

poisonous
EDIBLE       3768
POISONOUS    2168
Name: count, dtype: int64

<a name="division"></a>
<h1 style="color: #d60e8c;">División del conjunto de datos en datos en entrada y salida</h1>
<hr style="border: 0.5px solid #d60e8c;">


Obtenemos los atributos que caracterizan nuestro conjunto de datos:

In [11]:
df.columns

Index(['poisonous', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor',
       'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color',
       'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
       'stalk-surface-below-ring', 'stalk-color-above-ring',
       'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number',
       'ring-type', 'spore-print-color', 'population', 'habitat'],
      dtype='str')

Seleccionamos las columnas que utilizaremos como entradas y lo convertimos un <b>array de NumPy</b>, para poder utilizarlo con los algoritmos de <b>scikit-learn</b>:

In [ ]:
feature_df = df[['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach','exang', 'oldpeak', 'slope', 'ca', 'thal']]
#feature_df = df[df.columns[0:len(df_orig.columns) - 1] ]# Forma alternativa de obtener las columnas
X_readed = np.asarray(feature_df)
X_readed[0:5]
feature_df.shape

Y ahora los datos de salida, las clases:

In [ ]:
#y_readed = np.asarray(df[len(df_orig.columns) - 1]) # Forma alternativa de obtener la clase
y_readed = np.asarray(df['class'])
y_readed[0:5]

<a name="split"></a>
<h1 style="color: #d60e8c;">División en datos de entrenamiento y prueba</h1>
<hr style="border: 0.5px solid #d60e8c;">
El siguiente código, utiliza el 75% del conjunto de datos para entrenar el modelo y el 25% para test (valores por defecto si no se fijan el tamaño en los parámetros).

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X_readed, y_readed, random_state = 1)
print ('Train set:', X_train.shape,  y_train.shape)
print ('Test set:', X_test.shape,  y_test.shape)

NameError: name 'X_readed' is not defined

<a name="oversampling"></a>
<h1 style="color: #d60e8c;">Balanceo de datos</h1>
<hr style="border: 0.5px solid #d60e8c;">

En este ejemplo se utiliza la librería <a href="https://imbalanced-learn.org/stable/index.html"><b>imbalanced-learn</b></a> para el balanceo del conjunto de datos, añadiendo muestras en las clases minotitarias. Puedes consultar más información sobre los disintos métodos a aplicar en la documentación sobre <a href="https://imbalanced-learn.org/stable/over_sampling.html">oversampling</a> y <a href="https://imbalanced-learn.org/stable/under_sampling.html">undersampling</a>.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss

print ('Conjunto entrenamiento original', X_train.shape,  y_train.shape)
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))

# Balanceo de datos: ejemplo de oversampling
#sm = SMOTE(random_state=1)
#X_train, y_train= sm.fit_resample(X_train, y_train)

# Balanceo de datos: ejemplo de undersampling
sm = NearMiss()
X_train, y_train= sm.fit_resample(X_train, y_train)


print('\nBalanceado:', X_train.shape,  y_train.shape)
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))


<a name="scale"></a>
<h1 style="color: #d60e8c;">Normalización</h1>
<hr style="border: 0.5px solid #d60e8c;">

Los datos de entrada de este conjunto de datos tienen rangos de valores muy distintos, por lo que, es importante escalar los datos.

La librería <code>scikit-learn</code> dispone de un módulo para procesado de datos <code>preprocessing</code>. Vamos a utilizar <code>StandardScaler</code> (<a href="https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html#sklearn.preprocessing.StandardScaler">documentación</a>). Una vez entrenado, se puede utilizar posteriormente sobre nuevos datos.


In [ ]:
scaler = preprocessing.StandardScaler()
scaler.fit(X_train) # fit realiza los cálculos y los almacena

X_train = scaler.transform(X_train) # aplica los cálculos sobre el conjunto de datos de entrada para escalarlos
X_train[0:5]

<a name="pca"></a>
<h1 style="color: #d60e8c;">PCA</h1>
<hr style="border: 0.5px solid #d60e8c;">

Vamos utilizar el algoritmo <code>PCA</code> (<a href="https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html?highlight=pca#sklearn.decomposition.PCA">documentación del algoritmo</a>) implementado la librería <code>scikit-learn</code> sobre el conjunto de datos. Antes de utilizar PCA para extraer **n** componentes principales, lo utilizaresmos sin parámetros de forma que podemos analizar el porcentaje de varianza de todos los componentes utilizando "explained_variance_ratio_", que devuelve una lista, en este caso con 13 elementos porque vamos a usar los trece atributos, donde cada elemento de la lista indica el porcentaje de variabilidad que representa ese componente y, como utilizamos todos los componentes, la suma de todos los porcentajes, es decir, la suma de todos los elementos de la lista dará 1 como resultado.


In [ ]:
mypca = PCA()
mypca.fit(X_train)

mypca.explained_variance_ratio_

In [ ]:
mypca.explained_variance_ratio_.sum()  # Suma todos los elementos de la lista

A partir de <code>explained_variance_ratio_</code>, podemos calcular la varianza acumulada de los componentes:</li>   


In [ ]:
print("\n Varianza que aporta cada componente:")
variance = mypca.explained_variance_ratio_
print(variance)

print("\n Varianza acumulada:")
acumvar = variance.cumsum()

for i in range(len(acumvar)):
    print(f" {(i+1):2} componentes: {acumvar[i]:.8f} ")


<h3>Projección PCA con 2 componentes</h3>
Ahora vamos a utilizar PCA para extraer dos componentes y visualizar el resultado en una gráfica, así como calcular la pérdida de información respecto al conjunto original con los 13 atributos de entrada.

In [ ]:
mypca2 = PCA(n_components=2)
mypca2.fit(X_train)
values_proj2 = mypca2.transform(X_train)


Con la función <code>inverse_transform</code> se reconstruye el conjunto de datos de los <code>n</code> componentes principales (2 componentes, en este caso), y calcula la pérdida de información respecto al conjunto original. Para calcular esa pérdida, se puede calcular el promedio de las diferencias elevadas al cuadrado entre cada elemento del conjunto reconstruido y el original. Si sumamos el valor de pérdida obtenido al porcentaje de la varianza acumulada con dos componentes, vemos que el resultado es 1.

In [ ]:
X_projected2 = mypca2.inverse_transform(values_proj2)
loss2 = ((X_train - X_projected2) ** 2).mean()
print("Projection loss (2 components): " + str(loss2))

Vamos ahora a mostrar la gráfica de la distribución de los datos origninales con los dos primeros atributos y la proyección PCA con 2 componentes componentes principales. En ambos casos, se utilizan dos dimensiones:

In [ ]:
plt.figure()
plt.subplot(1,2,1) # 1 - numrows, 2 - numcols, 1 - index
plt.title("Datos originales con dos atributos")
plt.scatter(X_train[: ,0] , X_train[: ,1] ,marker='o' ,c=y_train)
plt.subplot(1,2,2) # 1 - numrows, 2 - numcols, 2 - index
plt.scatter(values_proj2[: ,0] , values_proj2[: ,1],marker='o' ,c=y_train)
plt.title("Proyección PCA con 2 componentes")
plt.subplots_adjust(right=1.9) # Distancia a la derecha
plt.show()


**Conclusión:**<br>
Si intentamos visualizar los datos con dos variables al azar, el resultado que obtenemos no nos proporciona información sobre la distribución en clases de los datos, como ocurre en la gráfica de la izquierda. Sin embargo, si visualizamos los datos en base a dos componentes, obtenemos información relevante y podemos visualizar ya cierta separación entre las clases, aunque, en este caso, tras ver los resultados de las varianza obtenida con cada componente, para obtener una mejor separación entre  clases necesitaríamos llegar a 8-9 componentes, pero no podemos visualizarlo gráficamente.

<h3>Projección con 11 componentes</h3>

In [ ]:
mypca11 = PCA(n_components=11)
mypca11.fit(X_train)
values_proj11 = mypca11.transform(X_train)

X_projected11 = mypca11.inverse_transform(values_proj11)
loss11 = ((X_train - X_projected11) ** 2).mean()

print("Projection loss (11 components): " + str(loss11))




<a name="classification"></a>
<div class="alert alert-block alert-info" 
     style="border: 0px solid #d60e8c; border-radius: 10px; background:#d60e8c; color: white;">
      <h2>EJERCICIO</h2>
    <hr style="border: 0.5px solid #ffffff;">
   <ul style="margin-bottom: 20px;">
       <li>
          Utiliza  dos algoritmos de clasificación supervisada sobre el conjunto de datos original y sobre dos proyecciones PCA (las que que consideres más relevantes para su análisis, una de ellas debe ser diferente a los ejemplos de este cuaderno). Para cada algoritmo de clasificación:
           <ol>
               <li>Crea el modelo, entrénalo y calcula su exactitud. Realiza varias pruebas con distintos parámetros para determinar
                  la mejor configuración. Muestra la matriz de confusión de cada uno.</li>
               <li> Realiza las pruebas con el conjunto datos sin balancear y balanceado. Analiza los resultados obtenidos en ambos casos. ¿Cómo afecta el balanceao a los resultados?¿Merece la pena utilizarlo?</li>               
               <li>Extrae conclusiones sobre los resultados obtenidos, comparando los resultados obtenidos sobre el conjunto de datos original con los 13 atributos y conjuntos de datos transformados con PCA. ¿Merece la pena utilizar PCA en este conjunto de datos? Razona la respuesta.</li>               
           </ol>           
       </li>    
    </ul>
    <font style="font-weight: bold;">Se recomienda realizar este ejercicio como un programa en Visual Studio Code. Servirá de preparación para las prácticas entregables.</font>
</div>

In [17]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm
from sklearn.model_selection import train_test_split

from sklearn import preprocessing
from sklearn.decomposition import PCA
import pandas as pd

from sklearn import preprocessing
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss

df_orig=pd.read_csv('mushroms1.csv', na_values=["?"])
df_orig.head()
df_orig.isnull().sum()
df=df_orig.dropna(axis=0)
df.isnull().sum()

#Pasamos de texto a numero
encoder = preprocessing.OrdinalEncoder(dtype=int)
df_encoded = encoder.fit_transform(df[['poisonous', "cap-shape","cap-surface","cap-color","bruises","odor",
                                      "gill-attachment","gill-spacing","gill-size","gill-color","stalk-shape","stalk-root",
                                      "stalk-surface-above-ring","stalk-surface-below-ring","stalk-color-above-ring",
                                      "stalk-color-below-ring","veil-type","veil-color","ring-number","ring-type",
                                      "spore-print-color","population","habitat"]])
encoder.categories_
df_encoded
df["poisonous"] = df_encoded[:,0]
df["cap-shape"] = df_encoded[:,1]
df["cap-surface"] = df_encoded[:,2]
df["cap-color"] = df_encoded[:,3]
df["bruises"] = df_encoded[:,4]
df["odor"] = df_encoded[:,5]
df["gill-attachment"] = df_encoded[:,6]
df["gill-spacing"] = df_encoded[:,7]
df["gill-size"] = df_encoded[:,8]
df["gill-color"] = df_encoded[:,9]
df["stalk-shape"] = df_encoded[:,10]
df["stalk-root"] = df_encoded[:,11]
df["stalk-surface-above-ring"] = df_encoded[:,12]
df["stalk-surface-below-ring"] = df_encoded[:,13]
df["stalk-color-above-ring"] = df_encoded[:,14]
df["stalk-color-below-ring"] = df_encoded[:,15]
df["veil-type"] = df_encoded[:,16]
df["veil-color"] = df_encoded[:,17]
df["ring-number"] = df_encoded[:,18]
df["ring-type"] = df_encoded[:,19]
df["spore-print-color"] = df_encoded[:,20]
df["population"] = df_encoded[:,21]
df["habitat"] = df_encoded[:,22]
df.head()

df.shape
df['poisonous'].value_counts()
df.columns
feature_df = df[df.columns[1:df.shape[1]]]
# Usando DataFrame Pandas
X = feature_df

# Usando array NumPy
#X = np.asarray(feature_df)
X[0:5] # solo para inspeccionar una muestra de X
# Usando DataFrame Pandas
y = df['poisonous']

# Usando array NumPy
#y = np.asarray(df['Class'])
y[0:5] # solo para inspeccionar una muestra de y

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=4)
print ('Training set:', X_train.shape,  y_train.shape)
print ('Test set:', X_test.shape,  y_test.shape)

print ('Conjunto entrenamiento original', X_train.shape,  y_train.shape)
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))

# Balanceo de datos: ejemplo de oversampling
#sm = SMOTE(random_state=1)
#X_train, y_train= sm.fit_resample(X_train, y_train)

# Balanceo de datos: ejemplo de undersampling
sm = NearMiss()
X_train, y_train= sm.fit_resample(X_train, y_train)


print('\nBalanceado:', X_train.shape,  y_train.shape)
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))

for i in range(1, 100):
    k = i

    # Creamos nuestra instancia del modelo
    neigh = KNeighborsClassifier(n_neighbors = k)

    ini = time.time() 
    #Entrenamiento del modelo, llamando a su método fit 
    neigh.fit(X_train,y_train)
    #print(f"Tiempo entrenamiento = {(time.time() - ini)*1000:.3f} ms") 

    ini = time.time() 
    y_predict_knn = neigh.predict(X_test)
    #print(f"Tiempo de predicción = {(time.time() - ini)*1000:.3f} ms") 

    y_predict_knn[0:5] # muestra del resultado de la predicción

    print("Exactitud media obtenida con k-NN para k={i}: ".format(i=i), neigh.score(X_test, y_test))

Training set: (4748, 22) (4748,)
Test set: (1188, 22) (1188,)
Conjunto entrenamiento original (4748, 22) (4748,)
{np.int64(0): np.int64(3008), np.int64(1): np.int64(1740)}

Balanceado: (3480, 22) (3480,)
{np.int64(0): np.int64(1740), np.int64(1): np.int64(1740)}
Exactitud media obtenida con k-NN para k=1:  0.9806397306397306
Exactitud media obtenida con k-NN para k=2:  0.9806397306397306
Exactitud media obtenida con k-NN para k=3:  0.9747474747474747
Exactitud media obtenida con k-NN para k=4:  0.9747474747474747
Exactitud media obtenida con k-NN para k=5:  0.9705387205387206
Exactitud media obtenida con k-NN para k=6:  0.9688552188552189
Exactitud media obtenida con k-NN para k=7:  0.9562289562289562
Exactitud media obtenida con k-NN para k=8:  0.9570707070707071
Exactitud media obtenida con k-NN para k=9:  0.9562289562289562
Exactitud media obtenida con k-NN para k=10:  0.9562289562289562
Exactitud media obtenida con k-NN para k=11:  0.9537037037037037
Exactitud media obtenida con k-

<br/><br/>
<hr style="border: 0.5px solid #d60e8c;">
<div style="text-align:right;">
MASTER UNIVERSITARIO EN INGENIERÍA INDUSTRIAL
</div>